In [ ]:
from risk_estimation.models.safety_layer import get_risk_estimator
from risk_estimation.datasets.frame_dropping import NoFrameDroppingPolicy
from risk_estimation.datasets.risk_dataloader import RiskEstimationDataset as D
from risk_estimation.datasets.risk_feature_extractor import *
from risk_estimation.datasets.frame_dropping import *
from risk_estimation.result_evaluator import benchmark_eval_save
from risk_estimation.scripts.result_img_save import sample_and_save_on_video, video_triplets_save
from video_embedding.models.video_embedder import VideoEmbedder
from video_embedding.utils import all_test_names, all_trial_names, set_session

from torch.utils.data import DataLoader

In [ ]:
features: str = "StampedLatentObservationsRiskLabels" # StampedLatent (h+alpha) (is default)
framedrop_policy: str = f"OnlyLabelledFramesDroppingPolicy" 
features = eval(features)
filter_samples_to_label_areas: bool = False # if True, only use samples in the label areas
if filter_samples_to_label_areas: framedrop_policy += f"Risk{skill_name}"
framedrop_policy = eval(framedrop_policy)
set_session("AE3")

In [ ]:
class SafetyLayer:
    def __init__(self, skill_name, approach, train_epoch=400):
        self.skill_name = skill_name
        self.video_embedder = VideoEmbedder(name=skill_name, latent_dim=12, nn_model="Autoencoder3")
        self.video_embedder.load_model()

        self.estimator = get_risk_estimator(
            approach, skill_name, features.xdim(12), self.video_embedder, "cautious", train_epoch, train_epoch
        )
        
    def train_valid(self, train_dataloader, test_dataloader, novel_dataloader=None):
        self.estimator.set_dataloaders_for_validation([test_dataloader], names=["test"])
        self.estimator.training_loop(train_dataloader, early_stop=True)
        self.estimator.save_model()

        benchmark_eval_save("Train_dataset", self.skill_name, train_dataloader.dataset, self.video_embedder, self.estimator)
        benchmark_eval_save("Test_dataset", self.skill_name, test_dataloader.dataset, self.video_embedder, self.estimator)
        if novel_dataloader is not None:
            benchmark_eval_save("Novel_Dataset", self.skill_name, novel_dataloader.dataset, self.video_embedder, self.estimator)

        for video_name in train_dataloader.dataset.video_names + test_dataloader.dataset.video_names:
            sample_and_save_on_video(video_name, self.video_embedder, self.estimator, features, train_dataloader, folder="autogen")
            video_triplets_save(video_name, self.video_embedder, folder="autogen")



In [ ]:
pick_gp = SafetyLayer(skill_name = "peg_pick404", approach = 'TwinGP')
pick_mlp = SafetyLayer(skill_name = "peg_pick404", approach = 'MLP2')
door_gp = SafetyLayer(skill_name = "peg_door404", approach = 'TwinGP')
door_mlp = SafetyLayer(skill_name = "peg_door404", approach = 'MLP2')
place_gp = SafetyLayer(skill_name = "peg_place404", approach = 'TwinGP')
place_mlp = SafetyLayer(skill_name = "peg_place404", approach = 'MLP2')

## Pick

In [ ]:
labeled_dataloader = D.load_dataloader(all_trial_names(pick_gp.skill_name), pick_gp.video_embedder, 64, framedrop_policy, features)
test_dataloader = D.load_dataloader(all_test_names(pick_gp.skill_name), pick_gp.video_embedder, 64, framedrop_policy, features)


RangedFramesDroppingPolicy.mintestcut = 200
RangedFramesDroppingPolicy.maxtestcut = 400
novel_dataloader = D.load_dataloader(all_test_names(pick_gp.skill_name), pick_gp.video_embedder, 64, RangedFramesDroppingPolicy, features)


nodrop_dataloader = D.load_dataloader(all_trial_names(pick_gp.skill_name), pick_gp.video_embedder, 64, NoFrameDroppingPolicy, features)
RangedFramesDroppingPolicy.mintestcut = 0
RangedFramesDroppingPolicy.maxtestcut = 110
dataloader_ood1 = D.load_dataloader(["peg_pick404_test_31"], pick_gp.video_embedder, 64, RangedFramesDroppingPolicy, features)
RangedFramesDroppingPolicy.mintestcut = 130
RangedFramesDroppingPolicy.maxtestcut = 700
dataloader_ood2 = D.load_dataloader(["peg_pick404_test_31"], pick_gp.video_embedder, 64, RangedFramesDroppingPolicy, features)
nodrop_train_dataloader_ood = DataLoader(nodrop_dataloader.dataset + dataloader_ood1.dataset + dataloader_ood2.dataset, batch_size=64)




1. Labelled Dataset

In [ ]:
pick_gp.train_valid(labeled_dataloader, test_dataloader, novel_dataloader)

2. No Drop Policy

In [ ]:

pick_gp.train_valid(nodrop_dataloader, test_dataloader, novel_dataloader)

3. Add OOD (special)

In [ ]:
pick_gp.train_valid(nodrop_train_dataloader_ood, test_dataloader, novel_dataloader)

MLP coparison

In [ ]:
pick_mlp.train_valid(nodrop_train_dataloader_ood, test_dataloader, novel_dataloader)

# Open Door

In [ ]:
labeled_dataloader = D.load_dataloader(all_trial_names(door_gp.skill_name), door_gp.video_embedder, 64, framedrop_policy, features)
test_dataloader = D.load_dataloader(all_test_names(door_gp.skill_name), door_gp.video_embedder, 64, framedrop_policy, features)
novel_dataloader = D.load_dataloader(all_test_names(door_gp.skill_name), door_gp.video_embedder, 64, framedrop_policy.novel(), features.novel())
nodrop_dataloader = D.load_dataloader(all_trial_names(door_gp.skill_name), door_gp.video_embedder, 64, NoFrameDroppingPolicy, features)

In [ ]:
door_gp.train_valid(labeled_dataloader, test_dataloader,novel_dataloader)

In [ ]:
door_gp.train_valid(nodrop_dataloader, test_dataloader, novel_dataloader)

In [ ]:
door_mlp.train_valid(nodrop_dataloader, test_dataloader, novel_dataloader)

# Place

In [ ]:
labeled_dataloader = D.load_dataloader(all_trial_names(place_gp.skill_name), place_gp.video_embedder, 64, framedrop_policy, features)
test_dataloader = D.load_dataloader(all_test_names(place_gp.skill_name), place_gp.video_embedder, 64, framedrop_policy, features)

RangedFramesDroppingPolicy.mintestcut = 200
RangedFramesDroppingPolicy.maxtestcut = 400

novel_dataloader = D.load_dataloader(all_test_names(place_gp.skill_name), place_gp.video_embedder, 64, RangedFramesDroppingPolicy, features)
nodrop_dataloader = D.load_dataloader(all_trial_names(place_gp.skill_name), place_gp.video_embedder, 64, NoFrameDroppingPolicy, features)

In [ ]:
# 1
place_gp.train_valid(labeled_dataloader, test_dataloader,novel_dataloader)

In [ ]:
# 2
door_gp.train_valid(nodrop_dataloader, test_dataloader, novel_dataloader)

In [ ]:
# 3
door_mlp.train_valid(nodrop_dataloader, test_dataloader, novel_dataloader)